In [12]:
import pandas as pd
import numpy as np
import os
import warnings

# Omitir advertencias de formato de Excel
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

# 1. Definir rutas (Asegúrate de que tus archivos se llamen exactamente así)
ruta_carpeta =  r"C:\Users\Andrea\OneDrive - Universidad Técnica Federico Santa María\Escritorio\MINERA"
file_fisico = os.path.join(ruta_carpeta, 'Exportaciones-Fisicas-por-Pais-de-destino-2005-2024a.xls')
file_valor = os.path.join(ruta_carpeta, 'Exportaciones-Chilenas-Valorizadas-por-Pais-de-Destino-2005-2024a.xls')
file_prod = os.path.join(ruta_carpeta, 'Produccion_1996-2024a.xls')

# =====================================================================
# FUNCIÓN ESPECIAL PARA ARCHIVOS JERÁRQUICOS DE ADUANAS/COCHILCO
# =====================================================================
def procesar_exportaciones(filepath, col_name):
    # Saltamos las primeras 4 filas de títulos del Excel
    df = pd.read_excel(filepath, skiprows=4)
    
    # 1. Rescatar el nombre del producto y "arrastrarlo" hacia abajo (ffill)
    df['Producto'] = np.where(df['País de Destino'].isna() & df['Unnamed: 2'].notna(), df['Unnamed: 2'], np.nan)
    df['Producto'] = df['Producto'].ffill()
    
    # 2. Filtrar solo las filas que sí tienen un país de destino
    df = df.dropna(subset=['País de Destino'])
    df = df.rename(columns={'País de Destino': 'Pais_Destino'})
    
    # 3. Eliminar columnas basuras de lectura
    cols_drop = [c for c in df.columns if str(c).startswith('Unnamed')]
    df = df.drop(columns=cols_drop)
    
    # 4. Tidy Data (Melt)
    df_melt = pd.melt(df, id_vars=['Producto', 'Pais_Destino'], var_name='Año', value_name=col_name)
    df_melt['Año'] = pd.to_numeric(df_melt['Año'], errors='coerce')
    df_melt = df_melt.dropna(subset=['Año'])
    df_melt['Año'] = df_melt['Año'].astype(int)
    
    # 5. Crear llave maestra (Ej: "YODO (28012000)" -> "YODO")
    df_melt['Llave_Producto'] = df_melt['Producto'].str.split(' ').str[0].str.strip().str.upper()
    return df_melt

def procesar_produccion(filepath):
    df = pd.read_excel(filepath, skiprows=3)
    df = df.rename(columns={'Unnamed: 0': 'Producto'})
    df = df.dropna(subset=['Producto'])
    
    df_melt = pd.melt(df, id_vars=['Producto'], var_name='Año', value_name='Produccion_Nacional_Ton')
    df_melt['Año'] = pd.to_numeric(df_melt['Año'], errors='coerce')
    df_melt = df_melt.dropna(subset=['Año'])
    df_melt['Año'] = df_melt['Año'].astype(int)
    
    df_melt['Llave_Producto'] = df_melt['Producto'].str.split(' ').str[0].str.strip().str.upper()
    df_melt = df_melt.drop(columns=['Producto']) # Evitamos columnas duplicadas en el cruce
    return df_melt

# =====================================================================
# EJECUCIÓN Y CRUCES
# =====================================================================
print("Procesando la estructura de los archivos Excel...")
df_valor = procesar_exportaciones(file_valor, 'Valor_FOB_Miles_USD')
df_fisico = procesar_exportaciones(file_fisico, 'Volumen_Fisico_Ton')
df_prod = procesar_produccion(file_prod)

print("Realizando cruces...")
# Paso 1: Inner Join entre Valorizadas y Físicas
df_integrado = pd.merge(df_valor, df_fisico, on=['Producto', 'Pais_Destino', 'Año', 'Llave_Producto'], how='inner')

# Paso 2: Left Join con Producción Nacional
df_final = pd.merge(df_integrado, df_prod, on=['Llave_Producto', 'Año'], how='left')

# Cálculo matemático del Precio Unitario
df_final['Volumen_Fisico_Ton'] = pd.to_numeric(df_final['Volumen_Fisico_Ton'], errors='coerce')
df_final['Valor_FOB_Miles_USD'] = pd.to_numeric(df_final['Valor_FOB_Miles_USD'], errors='coerce')

df_final['Precio_Unitario_USD_x_Ton'] = np.where(
    df_final['Volumen_Fisico_Ton'] > 0, 
    df_final['Valor_FOB_Miles_USD'] / df_final['Volumen_Fisico_Ton'], 
    np.nan
)

# Limpiamos la columna auxiliar de cruce y exportamos
df_final = df_final.drop(columns=['Llave_Producto'])

ruta_salida = os.path.join(ruta_carpeta, 'Mineria_Integrada_Consistente.csv')
df_final.to_csv(ruta_salida, index=False, sep=';', encoding='utf-8-sig')

print(f"¡Éxito total! Se cruzaron y guardaron {len(df_final)} registros.")

Procesando la estructura de los archivos Excel...
Realizando cruces...
¡Éxito total! Se cruzaron y guardaron 18540 registros.
